In [ ]:
# %config InlineBackend.figure_format = 'retina'

import ast
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import curve_fit
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

In [ ]:
# Load profiling and logprob data
size = "30"
base_path = f"../logs/frames_full_llamacpp_qwen3_{size}b"
# base_path = f"../logs/simpleqa_full_llamacpp_qwen3_{size}b"
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
regex = r"logprob=(-?\d+\.\d+(?:[eE][-+]?\d+)?)"
token_regex = r"ChatCompletionTokenLogprob\(token=(['\"])(.*?)\1, bytes="
all_logprobs = []
all_tokens = []
for i in range(max_folder + 1):
    with open(f"{base_path}/{i}/run_0/raw/trace.json", "r") as f:
        trace = json.load(f)
        trace = [t for t in trace if t["name"] == "ActionStep" and "logprob=" in t["attributes"]["output.value"]]
        trace = sorted(trace, key=lambda t: t["start_time"])
        logprobs = []
        tokens = []
        for t in trace:
            v = json.loads(t["attributes"]["output.value"])
            logprobs.append([
                float(re.search(regex, l).group(1)) for l in v["model_output_message"]["raw"]["logprobs"]
            ])
            tokens.append([
                re.search(token_regex, l).group(2) for l in v["model_output_message"]["raw"]["logprobs"]
            ])
        all_logprobs.append(logprobs)
        all_tokens.append(tokens)

input_path = f"../data/frames/profile_results_frames_full_llamacpp_qwen3_{size}b_judged.csv"
output_path = input_path[:-4] + "_classified.csv"
input_df = pd.read_csv(input_path)
is_correct = input_df["agent_output_eval"] == "CORRECT"
llm_eval = pd.read_csv(f"../data/frames/llm_frames_results_qwen3_30b_2507_judged.csv")
is_llm_correct = llm_eval["qwen3:30b-a3b-instruct-2507-q4_K_M_eval"] == "CORRECT"
num_wiki_links = llm_eval["wiki_links"].str.count(",") + 1
num_wiki_links = num_wiki_links.to_list()

# is_correct = pd.read_csv(f"../data/simpleqa/profile_results_simpleqa_full_llamacpp_qwen3_{size}b_judged.csv")["agent_output_eval"] == "CORRECT"
# llm_eval = pd.read_csv(f"../data/simpleqa/llm_simpleqa_results_qwen3_30b_2507_judged.csv")
# is_llm_correct = llm_eval["qwen3:30b-a3b-instruct-2507-q4_K_M_eval"] == "CORRECT"
# num_wiki_links = llm_eval["metadata"].apply(lambda x: len(set(ast.literal_eval(x)['urls'])))
# num_wiki_links = num_wiki_links.to_list()


# Only consider traces where the LLM doesn't already know the answer
all_logprobs = [all_logprobs[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
all_tokens = [all_tokens[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
is_correct = [is_correct[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
num_wiki_links = [num_wiki_links[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
valid_idx = [i for i in range(len(is_llm_correct)) if not is_llm_correct[i]]

In [ ]:
# Display barplot of number of successes/failures by number of steps taken to finish the task
incorrect_steps = [0] * 11
correct_steps = [0] * 11
for i, logprobs in enumerate(all_logprobs):
    n = len(logprobs)
    if is_correct[i]:
        correct_steps[n - 1] += 1
    else:
        incorrect_steps[n - 1] += 1

plt.figure(figsize=(8, 5))
for i in range(1, 12):
    plt.bar(i - 0.15, incorrect_steps[i - 1], width=0.25, color="red", label="Fail" if i == 1 else "")
    plt.bar(i + 0.15, correct_steps[i - 1], width=0.25, color="green", label="Success" if i == 1 else "")
plt.xticks(list(range(1, 12)))
plt.xlabel("Number of steps taken")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_logprobs_stats_by_step(func):
    step_logprobs_data = [[] for _ in range(11 * 2)]
    for i, logprobs in enumerate(all_logprobs):
        for step, step_logprobs in enumerate(logprobs):
            step_logprobs_data[step * 2 + int(is_correct[i])].append(func(step_logprobs))

    plt.figure(figsize=(8, 5))
    positions = []
    colors = []
    for i in range(1, 12):
        positions.extend([i-0.15, i+0.15])
        colors.extend(["red", "green"])
    
    for i in range(len(step_logprobs_data)):
        bp = plt.boxplot(
            [step_logprobs_data[i]],
            positions=[positions[i]],
            patch_artist=True,
            widths=0.25
        )

        box = bp['boxes'][0]
        box.set_facecolor(colors[i])
        box.set_linewidth(1)  # thinner border
        # Customize median line
        median = bp['medians'][0]
        median.set_color('black')

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    xticks = np.arange(1, 12, 1)
    ax.set_xlim(0, 12)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{y}" for y in xticks])
    
    plt.xlabel("Step")
    func_name = func.__name__.split(".")[-1]
    func_name = func_name[0].upper() + func_name[1:]
    plt.ylabel(f"{func_name} logprobs")
    plt.title(f"{func_name} logprobs vs Step")
    plt.show()

In [ ]:
plot_logprobs_stats_by_step(min)

In [ ]:
plot_logprobs_stats_by_step(np.mean)

In [ ]:
def plot_step_logprobs_line(func, diff=False):
    plt.figure(figsize=(8, 5))

    for i, logprobs in enumerate(all_logprobs):
        line = np.array([func(step_logprobs) for step_logprobs in logprobs])
        if diff:
            line = line[1:] - line[:-1]
        color = "green" if is_correct[i] else "red"
        style = "dotted" if is_correct[i] else "dotted"
        alpha = 0.8 if is_correct[i] else 0.8
        if len(line) == 1:
            plt.scatter([-0.1], line, color=color, s=20, zorder=3, alpha=alpha)
        else:
            plt.plot(range(len(line)), line, color=color, linestyle="--", alpha=0.6)

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    xticks = np.arange(0, 11, 1)
    ax.set_xlim(-1, 11)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{y}" for y in xticks])

    plt.xlabel("Step")
    func_name = func.__name__.split(".")[-1]
    func_name = func_name[0].upper() + func_name[1:]
    plt.ylabel(f"{func_name} logprobs{' change' if diff else ''}")
    plt.title(f"{func_name} logprobs{' change' if diff else ''} vs Step")

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

def plot_all_logprobs(step, moving_avg_window=None):
    for correct in [True, False]:
        plt.figure(figsize=(15, 5))

        count = 0
        for i, logprobs in enumerate(all_logprobs):
            if len(logprobs) <= step or is_correct[i] != correct:
                continue
            count += 1
            line = logprobs[step]
            if isinstance(moving_avg_window, int) and moving_avg_window > 1:
                line = np.convolve(line, np.ones(moving_avg_window)/moving_avg_window, mode='valid')
            color = "green" if is_correct[i] else "red"
            # style = "-" if is_correct[i] else "dotted"
            style = "-"
            alpha = 0.8 if is_correct[i] else 0.4
            if len(line) == 1:
                plt.scatter([-0.1], line, color=color, s=20, zorder=3, alpha=alpha)
            else:
                plt.plot(range(len(line)), line, color=color, linestyle=style, alpha=alpha)

        plt.xlabel("Token")
        plt.ylabel("Logprobs")
        plt.title(f"Logprobs vs Token ({count} traces)")

        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

def exp_saturation(x, L, k, c):
    return L * (1 - np.exp(-k * x)) + c

def fit_exp_saturation_curve(y):
    n = len(y)
    x = np.linspace(1, n, n)
    p0 = [max(y), 1.0, 0.0]
    params, _ = curve_fit(
        exp_saturation,
        x,
        y,
        p0=p0,
        bounds=([0.0, -np.inf, -np.inf], [1.0, np.inf, np.inf]),
        maxfev=10000,
    )
    return params

def plot_exp_saturation_curve(n, params, x_offset=0, **kwargs):
    L, k, c = params
    x = np.linspace(1, n, 100)
    y = exp_saturation(x, L, k, c)
    plt.plot(x_offset + x, y, **kwargs)

def plot_min_logprobs(
    start_step,
    end_step,
    num_logprob=10,
    exp=False,
    fit=False,
    normalize=False,
    trace_type=None,
):
    assert start_step <= end_step, "Start step must not be greater than end step"
    plt.figure(figsize=(16, 5))
    cnt = 0

    for i, logprobs in enumerate(all_logprobs):
        if len(logprobs) <= end_step:
            continue
        if trace_type is not None:
            if is_correct[i] != trace_type:
                continue
        cnt += 1
        # line = np.array([np.sort(step_logprobs)[:num_logprob] for step_logprobs in logprobs[:max_step]]).flatten()
        color = "green" if is_correct[i] else "red"
        # color = "#542788" if is_correct[i] else "#2D862D"
        # style = "-" if is_correct[i] else "dotted"
        style = "-" if not fit else "dotted"
        # alpha = 0.8 if is_correct[i] else 0.4
        alpha = 0.5
        # plt.scatter(range(len(line)), line, color=color, alpha=alpha)

        global_mean = np.mean([np.exp(v) for lp in logprobs[:end_step] for v in lp])
        global_std = np.std([np.exp(v) for lp in logprobs[:end_step] for v in lp])

        for step, step_logprobs in enumerate(logprobs[start_step-1:end_step]):
            if exp:
                step_logprobs = np.exp(step_logprobs)

            line = np.sort(step_logprobs)[:num_logprob]
            # line = np.quantile(step_logprobs, [0.0 + 0.05 * i for i in range(20)])

            if normalize:
                non_zero_step_logprobs = step_logprobs[step_logprobs < 1]
                # line = (line - np.mean(non_zero_step_logprobs)) / np.std(non_zero_step_logprobs)
                line = (line - global_mean) / global_std
                # line = (line - np.mean(line)) / np.std(line)

            if fit:
                params = fit_exp_saturation_curve(line)
                plot_exp_saturation_curve(num_logprob, params, x_offset=step * num_logprob, color=color, alpha=alpha, linestyle="-")

            start = step * num_logprob + 1
            plt.plot(range(start, start + len(line)), line, color=color, alpha=0.3, linestyle=style)

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    # xticks = np.arange(0, 11, 1)
    # ax.set_xlim(-1, 11)
    # ax.set_xticks(xticks)
    # ax.set_xticklabels([f"{y}" for y in xticks])

    plt.xlabel("Step")
    plt.ylabel("Logprobs" if not exp else "Probabilities")
    plt.title(f"Top {num_logprob} smallest token {'logprobs' if not exp else 'probabilities'} vs Step (n = {cnt})")

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

def plot_min_logprobs_tsne(
    start_step,
    end_step,
    num_logprobs=10,
    n_components=2,
    perplexity=30,
    lr=1000,
):
    tsne = TSNE(n_components=n_components, perplexity=perplexity, learning_rate=lr, random_state=42, init="pca")
    X = []
    y = []

    for i, logprobs in enumerate(all_logprobs):
        if len(logprobs) <= end_step:
            continue
        X.append([np.exp(v) for lp in logprobs[start_step-1:end_step] for v in np.sort(lp)[:num_logprobs]])
        y.append(int(is_correct[i]))

    X_embedded = tsne.fit_transform(np.array(X))
    y = np.array(y)

    plt.figure(figsize=(8, 6))
    plt.scatter(X_embedded[y == 1, 0], X_embedded[y == 1, 1], c="green", label="Success", alpha=0.7)
    plt.scatter(X_embedded[y == 0, 0], X_embedded[y == 0, 1], c="red", label="Fail", alpha=0.7)
    plt.legend()
    plt.title(f"t-SNE visualization of top {num_logprobs} logprob features (perplexity={perplexity}, lr={lr})")
    plt.xlabel('t-SNE dim 1')
    plt.ylabel('t-SNE dim 2')
    plt.show()

def plot_logprob_evolution(idx, start_step, end_step, num_logprobs=10):
    assert start_step <= end_step
    logprobs = all_logprobs[idx]

    data = np.vstack([np.sort(lp)[:num_logprobs] for lp in logprobs[start_step-1:end_step]])
    num_steps, length = data.shape

    # Axes:
    # t = time index inside each time series
    # s = external time step (one series per step)
    t = np.arange(length)
    steps = np.arange(num_steps) + 1

    T, S = np.meshgrid(t, steps)

    # ---- Plot 3D surface ----
    colors = plt.cm.tab20(np.linspace(0, 1, length))
    # colors = plt.cm.viridis(np.linspace(0, 1, length))
    facecolors = np.repeat(colors[np.newaxis, :, :], num_steps, axis=0)

    plt.close('all')
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    # ax.plot_surface(T, S, data, facecolors=facecolors, shade=False)

    for j in range(length):
        ax.plot(
            np.full(num_steps, t[j]),  # x-axis → time index (constant)
            steps, # y-axis → time step
            data[:, j],          # z-axis → actual values
            color=colors[j],
            linewidth=1
        )

        ax.scatter(
            np.full(num_steps, t[j]),
            steps,
            data[:, j],
            color=colors[j],
            s=80,  # size of the spheres (larger = bigger)
            marker='o',
            edgecolor='k',  # optional: add black edge for better visibility
            alpha=0.9
        )

    for i in range(num_steps):
        ax.plot(
            t,                   # all logprob indices (x-axis)
            np.full(length, steps[i]),# constant step (y-axis)
            data[i, :],           # all logprob values for step i (z-axis)
            color='gray',
            linestyle='dashed',
            linewidth=1
        )

    ax.set_xlabel("Logprob index")
    ax.set_xticks(t)
    ax.set_ylabel("Step")
    ax.set_yticks(steps)
    ax.set_zlabel("Logprob")
    ax.set_title(f"Logprob evolution for trace {idx} ({'success' if is_correct[idx] else 'fail'})")

    ax.view_init(elev=30, azim=-135)
    plt.show()


In [ ]:
plot_min_logprobs(1, 4, num_logprob=4, exp=True, fit=False, normalize=False, trace_type=None)
plot_min_logprobs(1, 4, num_logprob=3, exp=True, fit=False, normalize=True, trace_type=None)

In [ ]:
for perplexity in [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    for lr in [200]:
        plot_min_logprobs_tsne(3, 4, perplexity=perplexity, lr=lr)

In [ ]:
%matplotlib widget
plot_logprob_evolution(6, 1, 11)

In [ ]:
plot_all_logprobs(4, 7)

In [ ]:
plot_step_logprobs_line(np.min)

In [ ]:
def min_10(a):
    return np.sort(a)[8]
plot_step_logprobs_line(min_10)

In [ ]:
plot_step_logprobs_line(np.min, diff=True)

In [ ]:
def plot_num_steps_stats():
    for step in range(9):
        plt.figure(figsize=(8, 6))

        for correct in [True, False]:
            num_ideal_steps = []
            num_steps = []
            num_token_step_1 = []
            num_thought_tokens = []
            for i, tokens in enumerate(all_tokens):
                if len(tokens) <= step:
                    continue
                if is_correct[i] != correct:
                    continue
                num_steps.append(len(tokens) + (np.random.rand() - 0.5) / 4)
                num_ideal_steps.append(num_wiki_links[i] + (np.random.rand() - 0.5) / 4)
                num_token_step_1.append(len(tokens[step]))
                # num_thought_tokens.append(tokens[step].index("<code")/len(tokens[step]) if "<code" in tokens[step] else 1.0)
                num_thought_tokens.append(tokens[step].index("<code") if "<code" in tokens[step] else len(tokens[step]))
            color = "green" if correct else "red"
            
            plt.scatter(num_thought_tokens, num_steps, color=color, s=20, alpha=0.5)

        plt.xlabel(f"Number of tokens in step {step + 1}")
        plt.ylabel("Number of total steps")

        # plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

plot_num_steps_stats()

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

import difflib

def lcs(a: str, b: str) -> str:
    matcher = difflib.SequenceMatcher(None, a, b)
    match = matcher.find_longest_match(0, len(a), 0, len(b))
    if match.size == 0:
        return ""
    return a[match.a: match.a + match.size]

def normalize(arr):
    return (arr - np.mean(arr)) / np.std(arr)

# ------------------ 1) Prepare training data ------------------ #
def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # sum_logprobs = [np.sum(np.sort(p)[:3]) for p in logprobs[:clf_step]]
        # feature_vector.extend(sum_logprobs)
        
        # quantiles = [0.0, 0.025, 0.05, 0.075, 0.1, 0.125, 0.15, 0.175, 0.2, 0.225]
        # logprobs_quantile = [v for p in probs for v in np.quantile(p, quantiles)]
        # feature_vector.extend(logprobs_quantile)

        # min_logprobs_spread = [(min_logprobs[i + num_logprobs - 1] - min_logprobs[i]) for i in range(0, clf_step * num_logprobs, num_logprobs)]
        # feature_vector.extend(min_logprobs_spread)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        # num_ones = [np.sum(prob == 1) for prob in probs]
        # pct_ones = [o / n for o, n in zip(num_ones, num_tokens)]
        # print(pct_ones)
        # feature_vector.extend(pct_ones)

        # num_unique_tokens = [len(set(step_tokens)) / len(step_tokens) for step_tokens in tokens[:clf_step]]
        # feature_vector.extend(num_unique_tokens)

        # ratio_thought_tokens = [(step_tokens.index("<code")/len(step_tokens) if "<code" in step_tokens else 1.0) for step_tokens in tokens[:clf_step]]
        # feature_vector.extend(ratio_thought_tokens)

        num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        feature_vector.extend(num_thought_tokens)

        # has_wiki = ["wikipedia_search(" in "".join(step_tokens) for step_tokens in tokens[:clf_step]]
        # feature_vector.extend(has_wiki)

        # Length of longest common substring between current step and previous step
        for j in range(clf_step - 1, clf_step):
            cur_gen = "".join(tokens[j])
            prev_gen = "".join(tokens[j - 1])
            feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    X = np.array(X_features)
    y = np.array(y)

    # scaler = StandardScaler()
    # X = scaler.fit_transform(X)
    # poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
    # X = poly.fit_transform(X)

    return X, y, valid

def train_classifier(logprobs_data, tokens_data, labels, clf_step, n_folds=5, num_logprobs=10):
    X, y, valid = extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=num_logprobs)
    print(f"X dimension: {X.shape}")
    print(f"y positive: {y.mean().round(2)} ({y.sum()}/{X.shape[0]})")

    # ------------------ 2) Hyperparameter Search (Nested CV) ------------------ #
    estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    )

    param_grid = {
        "n_estimators": [10, 20, 50, 100, 200, 300],
        "learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
        "max_depth": [2, 3, 4, 5, 6],
        "min_child_weight": [1, 3, 5, 7, 10],
        # "subsample": [0.8, 0.9, 1.0],
        # "gamma": [0, 0.1, 0.2, 0.3, 0.5],
        # "reg_alpha": [0, 0.01, 0.1, 1, 5, 10],
        # "reg_lambda": [0.1, 0.5, 1, 5, 10]
    }

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42),
        n_jobs=-1,
    )

    search.fit(X, y)
    best_model = search.best_estimator_
    print("Best hyperparameters:", search.best_params_)

    # ------------------ 3) Out-of-Fold (OOF) Predictions ------------------ #
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_proba = np.zeros_like(y, dtype=float)

    for tr_idx, va_idx in cv.split(X, y):
        model = clone(best_model)
        model.fit(X[tr_idx], y[tr_idx])
        proba = model.predict_proba(X[va_idx])[:, 1]
        oof_proba[va_idx] = proba

    # ------------------ 4) Find Best Threshold Using OOF Predictions ------------------ #
    prec, rec, thr = precision_recall_curve(y, oof_proba)
    prec = prec[:-1]
    rec = rec[:-1]
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    best_idx = np.nanargmax(f1)
    best_threshold = thr[best_idx] if best_idx < len(thr) else 0.5
    

    # alpha = 0.0  # higher alpha -> prioritize recall more
    # # Compute custom score
    # score = alpha * rec + (1 - alpha) * prec
    # # Find best threshold
    # best_idx = np.argmax(score)
    # best_threshold = thr[best_idx] if best_idx < len(thr) else 1.0

    print("Best threshold:", round(float(best_threshold), 6))
    oof_preds = (oof_proba >= best_threshold).astype(int)

    # ------------------ 5) Evaluate Performance via CV ------------------ #
    print(f"CV num predicted positive: {oof_preds.sum()}")
    print("CV ROC AUC:", roc_auc_score(y, oof_proba))
    print("CV F1:", f1_score(y, oof_preds))
    print("CV Accuracy:", accuracy_score(y, oof_preds))
    print("CV Precision:", precision_score(y, oof_preds))
    print("CV Recall:", recall_score(y, oof_preds))

    return oof_proba, valid

    # # ------------------ 6) Final Model Training ------------------ #
    # final_model = clone(best_model).fit(X, y)
    # print("Final model trained on full dataset.")
    # return final_model

In [ ]:
labels = [not c for c in is_correct]
# for num_steps in [2, 3, 4, 5, 6, 7, 8, 9]:
for num_logprobs in [2, 4, 6, 8, 10]:
    for num_steps in [2, 3, 4, 5]:
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        oof_prob, valid = train_classifier(all_logprobs, all_tokens, labels, clf_step=num_steps, num_logprobs=num_logprobs)
        # all_oof_prob = [None] * len(input_df)
        # for j, i in enumerate(valid):
        #     all_oof_prob[valid_idx[i]] = oof_prob[j]
        # input_df[f"classifier_prob_S{num_steps}"] = all_oof_prob

    # input_df.to_csv(output_path, index=False)

In [ ]:
# Display a correlation heatmap of the features and label
labels = [c for c in is_correct]
X, y, _ = extract_features(all_logprobs, all_tokens, labels, clf_step=3)
plt.figure(figsize=(12, 8))
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
df['label'] = y
sns.heatmap(df.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()